In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Raw jhu incidence data into (days, num of states) dataframe

data_path = "data/1_raw/covid_data.csv"
raw_df = pd.read_csv(data_path)

states = ["az", "ca", "il", "md", "nj", "ny"]

incidence = (
    raw_df.loc[raw_df["geo_value"].isin(states) & (raw_df["signal"] == "confirmed_incidence_prop"), ["time_value", "geo_value", "value"]]
      .assign(time_value=lambda x: pd.to_datetime(x["time_value"])) 
      .pivot_table(index="time_value", columns="geo_value", values="value")
      .sort_index()
      .reindex(columns=states)                                        
)

if isinstance(incidence.columns, pd.MultiIndex):
    incidence.columns = incidence.columns.get_level_values(-1)

incidence.columns.name = None
incidence.index.name = None

#handle negative values
incidence = incidence.mask(incidence < 0, other = np.nan)
incidence = incidence.ffill()

# Fix the index to the range
full_idx = pd.date_range("2020-01-19", "2021-03-13", freq="D")
incidence = incidence.reindex(full_idx)
incidence = incidence.fillna(0)

incidence.to_csv("data/2_processed/covid_incidence.csv", index=True)